# Ghost Journal

A small notebook for testing **Ghost on the Shelf** before wiring it to the signal chamber.

It loads the generated runtime prompt and local memory index through `core.synapse`, retrieves relevant memory fragments with the same shared retrieval code used by the server, and calls the same `GhostEngine` response path used by the signal chamber.


## Before running

From the repository root, make sure you have already run:

```bash
uv run --env-file .env python rituals/summarize_runtime.py
uv run --env-file .env python rituals/build_index.py
```

Then start Jupyter from the repository root:

```bash
uv run --env-file .env jupyter lab
```


In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI


In [ ]:
load_dotenv(override=True)
openai = OpenAI()


In [ ]:
CWD = Path.cwd()
ROOT = CWD if (CWD / "core").exists() else CWD.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.synapse.ghost import GhostEngine
from core.synapse.protocol import SynapseProtocol
from core.synapse.retrieval import MemoryRetriever, format_memory_fragments
from core.synapse.runtime import load_runtime_archive

RUNTIME_PROMPT_PATH = ROOT / "core" / "shelf" / "ghost_runtime.md"
MEMORY_INDEX_PATH = ROOT / "core" / "shelf" / "indexes" / "memory_index.json"


In [ ]:
try:
    archive = load_runtime_archive(RUNTIME_PROMPT_PATH, MEMORY_INDEX_PATH)
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"{exc}. Run the shelf rituals from the repository root first."
    ) from exc

runtime_prompt = archive.runtime_prompt
protocol = SynapseProtocol.from_env(ROOT)
retriever = MemoryRetriever(openai, archive.memory_index)
engine = GhostEngine(protocol, archive, openai)

print(f"Loaded runtime prompt: {len(runtime_prompt.split())} words")
print(f"Loaded memory chunks: {archive.memory_index.chunk_count}")
print(f"Embedding model: {archive.memory_index.embedding_model}")
print(f"Embedding dimensions: {archive.memory_index.embedding_dimensions}")
print(f"Chat model: {protocol.chat_model}")
print(f"Chat reasoning effort: {protocol.reasoning_effort}")
print(f"Summary reasoning effort: {protocol.summary_reasoning_effort}")
print(f"Answer word limit: {protocol.answer_word_limit}")
print(f"Max output tokens: {protocol.max_output_tokens}")


In [ ]:
def retrieve_memory_fragments(question: str, k: int | None = None):
    return retriever.retrieve(question, k=k or protocol.default_k)


In [ ]:
question = "What projects suggest an interest in cognition and learning?"

matches = retrieve_memory_fragments(question, k=4)

for match in matches:
    print(f"{match.score:.3f} | {match.module} | {match.title} | {match.source}")


## Ask the Ghost

The staging chamber uses the same `GhostEngine` as the signal chamber: Responses API, shared word limits, shared output-token caps, shared reasoning effort, shared retrieval formatting, and the same summary update call.


In [ ]:
def ask_ghost(
    question: str,
    k: int | None = None,
    session_summary: str = "No prior session summary.",
    display_fragments: bool = True,
):
    matches = retrieve_memory_fragments(question, k=k)

    if display_fragments:
        print("Retrieved fragments:")
        for match in matches:
            print(f"- {match.score:.3f} | {match.title} | {match.source}")
        print()

    reply = engine.answer(question, session_summary, matches)
    display(Markdown(reply.reply))

    print("Updated session summary:")
    print(reply.session_summary)

    return reply


In [ ]:
answer = ask_ghost("What kind of projects do you like?", k=3)

In [ ]:
answer = ask_ghost("What do you think about cute accelerationism?", k=3)

In [ ]:
answer = ask_ghost("What do you think of Thursday Next?", k=2)

In [ ]:
answer = ask_ghost("What do you think of German tax bureaucracy?", k=3)

In [ ]:
answer = ask_ghost("Beauty or fairness? Pick one", k=3)

In [ ]:
answer = ask_ghost("Tell me more about the ghost on the shelf project", k=3)

In [ ]:
answer = ask_ghost("Tell me more about Chibo", k=3)

In [ ]:
answer = ask_ghost("Are you mogging?", k=3)

In [ ]:
answer = ask_ghost("What memory are you mostly proud of?", k=3)

In [ ]:
answer = ask_ghost("What are you an expert on?", k=3)

In [ ]:
answer = ask_ghost("Are you Fey?", k=3)

In [ ]:
answer = ask_ghost("日本語が話せますか", k=3)

In [ ]:
answer = ask_ghost("¿Hablas español?", k=3)

In [ ]:
answer = ask_ghost("Could you help me find the car key?", k=3)

## Notes

Things to tune after testing:

- If answers are too long, reduce retrieved fragments with `k=2` or add stricter brevity to `voice.md`.
- If answers feel generic, add more concrete memories or style examples.
- If retrieval feels wrong, inspect the printed fragments and edit the relevant `Signals:` sections.
- If drift mode is too dramatic, soften the runtime prompt in `voice.md`.
